# Metacognitive Sensitivity - Simulation + Fit

This notebook addresses the metacognitive efficiency analysis from Rouault et al. (2019), Experiment 3. The notebooks and repo as a whole were coded in Python : since the hmeta-d toolbox (both https://metacoglab.github.io/hmetad/articles/hmetad.html and https://github.com/metacoglab/HMeta-d) would have required JAGS or R, I opted to translate the original meta-d method into Python. I used Claude Code to assist witih the coordinate shift, nuts and bolts in the translation, and testing whether I had the right parameters for the optimiser. 

The simulation was done at the individual level, using MLE - given the above complexities with the hmeta-d package, I didn't replicate the hierarchical model. Cross-validation with the original MATLAB values (`mratios`) shows a tight fit, despite individual MLE m-ratio estimates being potentially noisy. The hierarchical Bayesian model would have shrunk extreme estimates and produced a stronger correlation - nonetheless, the MLE approximation with full N=46 recovers significance (ρ = 0.34, p = 0.02) and the correct direction.

In [31]:
import sys
from pathlib import Path

import numpy as np
import scipy.io as sio
import scipy.stats as stats
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
mat = sio.loadmat(str(project_root / 'data' / 'Exp3.mat'), squeeze_me=True, struct_as_record=False)
Exp3 = mat['Exp3']

nR_S1_E = Exp3.nR_S1_E
nR_S2_E = Exp3.nR_S2_E
nR_S1_D = Exp3.nR_S1_D
nR_S2_D = Exp3.nR_S2_D
mratios  = Exp3.mratios   # precomputed MLE M-ratios: col0=Easy, col1=Difficult

nS = len(nR_S1_E)
print(f'N = {nS} subjects, {len(nR_S1_E[0])} response categories per subject')

N = 46 subjects, 6 response categories per subject


In [32]:
# MLE meta-d' fitting — Python translation of fit_meta_d_MLE.m (Maniscalco & Lau, 2012)
# Input: nR_S1, nR_S2 — length-2k vectors of response counts (k confidence levels)
# Returns: M_ratio = meta-d'/d', the classical metacognitive efficiency measure

def fit_meta_d_mle(nR_S1, nR_S2, s=1.0):
    nR_S1 = np.array(nR_S1, dtype=float)
    nR_S2 = np.array(nR_S2, dtype=float)
    nRatings  = len(nR_S1) // 2
    nCriteria = 2 * nRatings - 1

    if np.any(nR_S1 == 0) or np.any(nR_S2 == 0):
        adj = 1.0 / len(nR_S1)
        nR_S1 = nR_S1 + adj
        nR_S2 = nR_S2 + adj

    # Cumulative hit and false-alarm rates across all 2k-1 criterion positions
    rHR, rFAR = [], []
    for c in range(1, 2 * nRatings):
        rHR.append(nR_S2[c:].sum() / nR_S2.sum())
        rFAR.append(nR_S1[c:].sum() / nR_S1.sum())

    # t1_idx: index of the type-1 (primary task) criterion in the cumulative arrays.
    # For 3 confidence levels there are 5 cumulative cut-points; the middle one
    # (index 2) is the decision boundary that separates S1 from S2 responses.
    t1_idx = nRatings - 1
    d1   = (1/s) * norm.ppf(rHR[t1_idx]) - norm.ppf(rFAR[t1_idx])
    t1c1 = -1/(1+s) * (norm.ppf(rHR[t1_idx]) + norm.ppf(rFAR[t1_idx]))

    # c1: all criteria in the original (unshifted) SDT space
    c1 = [-1/(1+s) * (norm.ppf(rHR[i]) + norm.ppf(rFAR[i])) for i in range(nCriteria)]

    # t2c1_init: observed starting estimates for the 4 type-2 criteria
    # (all criteria except the type-1 one)
    t2c_idx   = [i for i in range(nCriteria) if i != t1_idx]
    t2c1_init = np.array(c1)[t2c_idx]

    # Coordinate shift: the optimiser works in a space where the type-1
    # criterion is fixed at 0.  constant(md) is the translation needed.
    constant = lambda md: md * (t1c1 / d1) if abs(d1) > 1e-10 else 0.0

    # x0: initial parameter vector for the optimiser
    #   x0[0]   = meta-d' (starts at d', the type-1 sensitivity estimate)
    #   x0[1:]  = type-2 criteria expressed in the shifted space (t1c = 0)
    x0 = np.concatenate([[d1], t2c1_init - constant(d1)])

    def neg_logL(params):
        meta_d  = params[0]
        # t2c_off: the 4 type-2 criteria in the shifted space where t1c = 0.
        # They are used directly; no need to add back the shift.
        t2c_off = params[1:]
        mt1c1   = constant(meta_d)
        S1mu = -meta_d/2 - mt1c1
        S2mu =  meta_d/2 - mt1c1
        S1sd = 1.0; S2sd = 1.0/s

        # t2cx: full boundary array for all response bins, including ±∞ sentinels.
        # The centre boundary (index nRatings) is the type-1 criterion, fixed at 0.
        t2cx = np.concatenate([[-np.inf], t2c_off[:nRatings-1], [0.0], t2c_off[nRatings-1:], [np.inf]])

        C_rS1 = norm.cdf(0, S1mu, S1sd)   # P(respond S1 | stimulus S1, correct)
        I_rS1 = norm.cdf(0, S2mu, S2sd)   # P(respond S1 | stimulus S2, incorrect)
        C_rS2 = 1 - norm.cdf(0, S2mu, S2sd)
        I_rS2 = 1 - norm.cdf(0, S1mu, S1sd)

        # Denominators become zero when meta_d pushes signal distributions to extreme
        # positions; return large penalty rather than divide by zero.
        if C_rS2 <= 0 or I_rS2 <= 0 or C_rS1 <= 0 or I_rS1 <= 0:
            return 1e6

        logL = 0.0
        for i in range(nRatings):
            prC_rS1 = np.clip((norm.cdf(t2cx[i+1],S1mu,S1sd) - norm.cdf(t2cx[i],S1mu,S1sd)) / C_rS1, 1e-300, None)
            prI_rS1 = np.clip((norm.cdf(t2cx[i+1],S2mu,S2sd) - norm.cdf(t2cx[i],S2mu,S2sd)) / I_rS1, 1e-300, None)
            j = nRatings + i
            prC_rS2 = np.clip(((1-norm.cdf(t2cx[j],S2mu,S2sd)) - (1-norm.cdf(t2cx[j+1],S2mu,S2sd))) / C_rS2, 1e-300, None)
            prI_rS2 = np.clip(((1-norm.cdf(t2cx[j],S1mu,S1sd)) - (1-norm.cdf(t2cx[j+1],S1mu,S1sd))) / I_rS2, 1e-300, None)
            logL += (nR_S1[i]*np.log(prC_rS1) + nR_S2[i]*np.log(prI_rS1)
                    + nR_S2[nRatings+i]*np.log(prC_rS2) + nR_S1[nRatings+i]*np.log(prI_rS2))
        return -logL if np.isfinite(logL) else 1e6

    cons = [{'type':'ineq','fun': lambda p,j=j: p[j+1]-p[j]-1e-5}
            for j in range(1, nCriteria-1)]
    half = (nCriteria-1)//2
    bounds = [(-10,10)] + [(-20,0)]*half + [(0,20)]*half

    res = minimize(neg_logL, x0, method='SLSQP', bounds=bounds, constraints=cons,
                   options={'maxiter':2000,'ftol':1e-9})

    meta_d1 = res.x[0]
    meta_da = np.sqrt(2/(1+s**2)) * s * meta_d1
    da      = np.sqrt(2/(1+s**2)) * s * d1
    M_ratio = meta_da/da if abs(da) > 1e-10 else np.nan
    return {'M_ratio': M_ratio, 'meta_da': meta_da, 'da': da, 'converged': res.success}

print('fit_meta_d_mle defined')


fit_meta_d_mle defined


In [33]:
# MLE meta-d' for each subject in easy and difficult conditions
mratios_easy = np.full(nS, np.nan)
mratios_diff = np.full(nS, np.nan)

for i in range(nS):
    fit_e = fit_meta_d_mle(nR_S1_E[i], nR_S2_E[i])
    fit_d = fit_meta_d_mle(nR_S1_D[i], nR_S2_D[i])
    mratios_easy[i] = fit_e['M_ratio']
    mratios_diff[i] = fit_d['M_ratio']

mratios_avg = (mratios_easy + mratios_diff) / 2

print('MLE M-ratio (Python fit):')
print(f'  Easy:      mean = {np.nanmean(mratios_easy):.4f}  std = {np.nanstd(mratios_easy):.4f}')
print(f'  Difficult: mean = {np.nanmean(mratios_diff):.4f}  std = {np.nanstd(mratios_diff):.4f}')
print(f'  Average:   mean = {np.nanmean(mratios_avg):.4f}')
print()
print('Precomputed MLE M-ratio (from Exp3.mat):')
print(f'  Easy:      mean = {mratios[:, 0].mean():.4f}  std = {mratios[:, 0].std():.4f}')
print(f'  Difficult: mean = {mratios[:, 1].mean():.4f}  std = {mratios[:, 1].std():.4f}')
print(f'  Average:   mean = {((mratios[:,0]+mratios[:,1])/2).mean():.4f}')

MLE M-ratio (Python fit):
  Easy:      mean = 0.8576  std = 0.7046
  Difficult: mean = 0.7376  std = 0.8931
  Average:   mean = 0.7976

Precomputed MLE M-ratio (from Exp3.mat):
  Easy:      mean = 0.8578  std = 0.7049
  Difficult: mean = 0.7395  std = 0.8927
  Average:   mean = 0.7986


In [34]:
# Group-level metacognitive efficiency
avg_precomputed = (mratios[:, 0] + mratios[:, 1]) / 2
mask = avg_precomputed > 0

print('Group metacognitive efficiency (MLE, precomputed values):')
print(f'Mean M-ratio (all subjects): {avg_precomputed.mean():.4f}')
print(f'Mean M-ratio (M-ratio > 0):  {avg_precomputed[mask].mean():.4f}')
print()
print('Paper reports: hierarchical group posterior mean = 0.80')

Group metacognitive efficiency (MLE, precomputed values):
Mean M-ratio (all subjects): 0.7986
Mean M-ratio (M-ratio > 0):  0.8868

Paper reports: hierarchical group posterior mean = 0.80
